In [1]:
import pandas as pd

In [2]:
base_path = '../../../data/monitoring/hcb junio 2026/'

Enlace a seguimiento financiero y macro: https://icbfgob.sharepoint.com/:x:/r/sites/RepositorioPrimeraInfancia2/_layouts/15/Doc.aspx?sourcedoc=%7BA9BA45C5-1435-4509-99D7-E69E5E86CB93%7D&file=validaci%C3%B3n_hcb.xlsx&action=default&mobileredirect=true

In [3]:
validacion_macro = pd.read_excel(base_path + 'validación_hcb.xlsx', header=1)

In [4]:
validacion_macro.head(2)

,Regional,Financiera,Macro
0,Bogota D.C.,True,True
1,Caquetá,True,False


Enlace a seguimiento ranking: https://icbfgob-my.sharepoint.com/shared?id=%2Fsites%2FRepositorioPrimeraInfancia2%2FDocumentos%20compartidos%2F2026%2F07%2E%20Equipo%20de%20Planeaci%C3%B3n%20Estrategica%2FAdiciones%20HCB%202026%2FRanking&listurl=https%3A%2F%2Ficbfgob%2Esharepoint%2Ecom%2Fsites%2FRepositorioPrimeraInfancia2%2FDocumentos%20compartidos&viewid=f28ff8fc%2D9784%2D4f91%2Db6aa%2D075937e262df

In [5]:
import os
import glob

In [6]:
ranking_path = base_path + 'Ranking/'
ranking_files = glob.glob(ranking_path + "ranking_*.xlsx")
print(f"Archivos de ranking encontrados: {len(ranking_files)}")

Archivos de ranking encontrados: 29


In [7]:
ranking_dfs = []
for f in ranking_files:
    df = pd.read_excel(f)
    ranking_dfs.append(df)

ranking_all = pd.concat(ranking_dfs, ignore_index=True)
print(f"Total registros combinados: {len(ranking_all)}")
ranking_all.head(2)

Total registros combinados: 1450


,Regional UDS,No. Contrato,PACCO,CDP,SITCO,Estudios Previos,Documentos del Operador,Radicación Jurídica,SECOP,Póliza,RP,Referencia de Contrato,Categoria EAS
0,Antioquia,5020432025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Antioquia,5020392025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
cols_to_check = [c for c in ranking_all.columns if c not in ('Regional UDS', 'No. Contrato')]

summary = []
for c in cols_to_check:
    total = len(ranking_all)
    filled = ranking_all[c].notna().sum()
    pct = round(filled / total * 100, 1)
    summary.append({'Columna': c, 'Diligenciado': filled, 'Total': total, '% Diligenciamiento': pct})

df_summary = pd.DataFrame(summary).sort_values('% Diligenciamiento', ascending=False)
df_summary

,Columna,Diligenciado,Total,% Diligenciamiento
0,PACCO,0,1450,0.0
1,CDP,0,1450,0.0
2,SITCO,0,1450,0.0
3,Estudios Previos,0,1450,0.0
4,Documentos del Operador,0,1450,0.0
5,Radicación Jurídica,0,1450,0.0
6,SECOP,0,1450,0.0
7,Póliza,0,1450,0.0
8,RP,0,1450,0.0
9,Referencia de Contrato,0,1450,0.0


### % Diligenciamiento por Regional

In [9]:
pivot = ranking_all.groupby('Regional UDS')[cols_to_check].apply(
    lambda x: x.notna().mean() * 100
).round(1)

pivot.insert(0, 'Total Contratos', ranking_all.groupby('Regional UDS').size())
pivot

,Total Contratos,PACCO,CDP,SITCO,Estudios Previos,Documentos del Operador,Radicación Jurídica,SECOP,Póliza,RP,Referencia de Contrato,Categoria EAS
Regional UDS,,,,,,,,,,,,
Antioquia,107,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Arauca,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Atlántico,73,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Bogota D.C.,199,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Bolívar,108,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Boyacá,61,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Caldas,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Caquetá,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cauca,71,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
val_legible = validacion_macro.copy()
for c in ['Financiera', 'Macro']:
    val_legible[c] = val_legible[c].map({True: 'Completo', False: 'Pendiente'})

val_legible

,Regional,Financiera,Macro
0,Bogota D.C.,Completo,Completo
1,Caquetá,Completo,Pendiente
2,Tolima,Completo,Completo
3,Antioquia,Pendiente,Pendiente
4,Arauca,Completo,Completo
5,Bolívar,Pendiente,Pendiente
6,Boyacá,Pendiente,Pendiente
7,Caldas,Pendiente,Pendiente
8,Cauca,Pendiente,Pendiente
9,Cesar,Completo,Pendiente


In [11]:
output_file = base_path + 'Avance_diligenciamiento_hcb.xlsx'

with pd.ExcelWriter(output_file) as writer:
    val_legible.to_excel(writer, sheet_name='Validación Gral', index=False)
    df_summary.to_excel(writer, sheet_name='Resumen ranking', index=False)
    pivot.reset_index().to_excel(writer, sheet_name='Ranking por regional', index=False)

print(f"Excel guardado en: {output_file}")

Excel guardado en: ../../../data/monitoring/hcb junio 2026/Avance_diligenciamiento_hcb.xlsx
